Si vous générer du texte avec des RNN / FNN, vous pouvez utilisez un modèle word2vec pour générer les plongements des tokens à l‘entrée, ou alors réapprendre les plongements pendant l’entraînement avec une couche d’embedding. Vous pouvez aussi utiliser tf-idf même si cela est moins courant. Vous n’êtes pas obligé d’utiliser toutes les méthodes de plongements ici, choisissez-en une.

Pour classifier et générer du texte avec un transformer, je vous recommande la bibliothèque Transformers de HuggingFace (et notamment le Trainer pour l'entrainement des modèles : https://huggingface.co/docs/transformers/main/en/trainer). Vous pouvez utiliser un petit modèle disponible sur leur site. Ne prenez pas un modèle à plus d’un milliard de paramètres si vous n’avez pas de bons GPUs, ce qui m’intéresse est comment vous aborder un problème, pas d’avoir les meilleurs performances possibles.

## Introduction (RNN)

- Plongement des tokens : Word2vec
- Plogement des documents : TF-IDF

### Evaluation

- BLEU/ROUGE Score
- BERT Score

### Bechmark

| Modèle                            | BLEU Score | ROUGE Score | BERTScore |
|----------------------------------|------------|-------------|-----------|
| **GRU (Word2Vec)**               | 0.0261     | 0.1100      | 0.4550    |
| **LSTM (Word2Vec)**              | 0.0215     | 0.1209      | 0.4348    |
| **GRU Bidirectionnel (Word2Vec)**| 0.0249     | 0.1161      | 0.4499    |
| **LSTM Bidirectionnel (Word2Vec)**| 0.0254     | 0.1143      | 0.4529    |

## Common code

In [2]:
import pandas as pd
from nltk.tokenize import word_tokenize

In [3]:
df_all = pd.read_csv("MovieDataThread.csv")
df = df_all.sample(n=1000, random_state=42) # Remove this line to use the entire dataset
display(df["Script"].head())


37439    Okay, go ahead.\n My name is Sidney Westerfeld...
11898    1\n A plague has descended upon Earth.\n In on...
24628    Hi, Leo.\n Say, kid, if these flowers ain't | ...
26980    1\n Under the light\n Of the full moon\n Shimm...
35203    Spend all day with us.\n There are two--\n par...
Name: Script, dtype: object

In [4]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [5]:
import re

def script_tokenize(text):
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\n", " NEWLINE_TOKEN ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip().split()

def custom_tokenize(text):
    first_tokenize = script_tokenize(text)
    entokenized = " ".join(first_tokenize)

    second_tokenize = word_tokenize(entokenized)
    return second_tokenize

In [6]:
from gensim.models import Word2Vec

# Tokeniser tous les scripts
tokenized_scripts = [custom_tokenize(script) for script in df_train["Script"]]

# Entraînement Word2Vec
w2v_model = Word2Vec(sentences=tokenized_scripts, vector_size=100, window=5, min_count=2, workers=4)

# Sauvegarder le modèle
w2v_model.save("word2vec_movie_scripts.model")

In [7]:
import torch

# Add padding (for too short sequences) and unknown tokens (for out-of-vocabulary words) by default
word_to_idx = {'<PAD>': 0, '<UNK>': 1}
for i, word in enumerate(w2v_model.wv.index_to_key):
    word_to_idx[word] = i + 2
idx_to_word = {i: w for w, i in word_to_idx.items()}

# Create the embedding matrix, where each row corresponds to a word in the vocabulary and each column corresponds to a dimension of the embedding
embedding_matrix = torch.zeros((len(word_to_idx), 100))
for word, i in word_to_idx.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = torch.tensor(w2v_model.wv[word])


In [8]:
def encode_sequence(tokens):
    return [word_to_idx.get(t, 1) for t in tokens]

def prepare_sequences(tokenized_texts, seq_len=20):
    X, y = [], []
    for tokens in tokenized_texts:
        encoded = encode_sequence(tokens)
        for i in range(len(encoded) - seq_len):
            X.append(encoded[i:i+seq_len])
            y.append(encoded[i+seq_len])
    return torch.tensor(X), torch.tensor(y)

# Limit the dataset to 50,000 sequences
# 1 sequence = 20 tokens
# X = 20 tokens, y = 21st token
X_train, y_train = prepare_sequences(tokenized_scripts, seq_len=20)
X_train = X_train[:50000]
y_train = y_train[:50000]

In [15]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
import re
import string

def calculate_bleu(reference, candidate):
    """
    Calculate BLEU score (it measures the similarity between two sentences)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU score
    """
    smoothing_function = SmoothingFunction().method4
    bleu_score = sentence_bleu([reference], candidate, smoothing_function=smoothing_function)
    return bleu_score

def calculate_rouge(reference, candidate):
    """
    Calculate ROUGE-L score (it calculates the longest common subsequence)
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The ROUGE score
    """
    rouge = Rouge()
    scores = rouge.get_scores(candidate, reference)
    return scores[0]['rouge-l']['f']

def preprocess_text(text):
    """
    Preprocess the text by removing punctuation and converting to lowercase
    :param text: The input text
    :return: The preprocessed text
    """
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = text.lower()
    return text

def calculate_scores(reference, candidate):
    """
    Calculate BLEU and ROUGE scores for the given reference and candidate sentences
    :param reference: The reference sentence
    :param candidate: The candidate sentence
    :return: The BLEU and ROUGE scores
    """
    # Preprocess the text
    reference = preprocess_text(reference)
    candidate = preprocess_text(candidate)

    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    bleu_score = calculate_bleu(reference_tokens, candidate_tokens)
    rouge_score = calculate_rouge(reference, candidate)

    return bleu_score, rouge_score

def calculate_perplexity(model, inputs, targets, criterion):
    model.eval()
    with torch.no_grad():
        outputs = model(inputs)         # (1, seq_len, vocab_size)
        outputs = outputs.view(-1, outputs.size(-1))  # (seq_len, vocab_size)
        targets = targets.view(-1)   
        loss = criterion(outputs, targets)
    return torch.exp(loss).item()

## RNN (GRU + Word2vec)
### Class

In [8]:
import torch.nn as nn

# Define the RNN model
# The model is a simple GRU with an embedding layer (Word2Vec embeddings)
class ScriptRNN(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        self.rnn = nn.GRU(embedding_matrix.size(1), hidden_size, batch_first=True, num_layers=2, dropout=0.2)
        self.fc = nn.Linear(hidden_size, embedding_matrix.size(0))

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

### Train

In [9]:
import torch
from torch.utils.data import DataLoader, TensorDataset
# Check if MPS is available
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

# Model, loss, optimizer
model = ScriptRNN(embedding_matrix, hidden_size=64).to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=word_to_idx['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Dataloader
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_model.pth"):
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("New best model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_script_rnn.pth")


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/10 | Loss: 5.3053
New best model saved.
Epoch 2/10 | Loss: 4.1566
New best model saved.
Epoch 3/10 | Loss: 3.8574
New best model saved.
Epoch 4/10 | Loss: 3.6648
New best model saved.
Epoch 5/10 | Loss: 3.5132
New best model saved.
Epoch 6/10 | Loss: 3.3845
New best model saved.
Epoch 7/10 | Loss: 3.2707
New best model saved.
Epoch 8/10 | Loss: 3.1681
New best model saved.
Epoch 9/10 | Loss: 3.0758
New best model saved.
Epoch 10/10 | Loss: 2.9943
New best model saved.


### Text generation

In [10]:
import torch.nn.functional as F

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0, device="cpu"):
    model.eval()
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()
    for _ in range(max_len):
        # Context = last seq_len tokens of the generated text
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
            
        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        next_word = idx_to_word.get(next_idx, "<UNK>")

        if next_word == "<PAD>": break
        tokens.append(next_word)
        generated.append(next_word)
    return " ".join(generated)

### Evaluation

In [12]:
import time
from bert_score import BERTScorer

bleu_scores, rouge_scores, bert_scores = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_SIZE = 100

scorer = BERTScorer(lang="en", rescale_with_baseline=True, model_type="bert-base-uncased")

print("Generating text and calculating scores...")
for script in df_test["Script"]:
    print("=" * 50)
    start_time = time.time()

    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)

    generated_text = generate_text(
        model, seed_text, word_to_idx, idx_to_word,
        max_len=GENERATE_LIMIT_SIZE,
        seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
        device=device,
        temperature=0.8
    )

    generated_tokens = custom_tokenize(generated_text)
    new_tokens = generated_tokens[len(seed_tokens):]
    generated = " ".join(new_tokens)

    reference_tokens = custom_tokenize(script)
    reference = " ".join(reference_tokens)[len(seed_text):len(seed_text) + len(generated)]

    print(f"Seed:\n{seed_text}")
    print("-" * 50)
    print(f"Reference:\n{reference}")
    print("-" * 50)
    print(f"Generated:\n{generated}")
    print("-" * 50)
    print(f"Time taken: {time.time() - start_time:.2f} seconds")

    bleu, rouge = calculate_scores(reference, generated)
    P, R, F1 = scorer.score([generated], [reference])
    bert = F1.mean().item()

    # Perplexity (attention: targets doivent être décalés !)
    inputs = torch.tensor([[word_to_idx.get(tok, word_to_idx["<UNK>"]) for tok in seed_tokens]], dtype=torch.long).to(device)
    targets = torch.tensor([[word_to_idx.get(tok, word_to_idx["<UNK>"]) for tok in new_tokens]], dtype=torch.long).to(device)
    targets = targets[:, :inputs.shape[1]]  # Align shapes
    bleu_scores.append(bleu)
    rouge_scores.append(rouge)
    bert_scores.append(bert)

    print(f"BLEU score: {bleu:.4f}")
    print(f"ROUGE score: {rouge:.4f}")
    print(f"BERTScore: {bert:.4f}")

print("=" * 50)
print("Average BLEU score:", sum(bleu_scores) / len(bleu_scores))
print("Average ROUGE score:", sum(rouge_scores) / len(rouge_scores))
print("Average BERTScore:", sum(bert_scores) / len(bert_scores))

/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Generating text and calculating scores...
Seed:
1 NEWLINE_TOKEN Delhi flight ! NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN Boarding is closed . NEWLINE_TOKEN Listen , I know ! NEWLINE_TOKEN I am so sorry , I got delayed . NEWLINE_TOKEN Just do something . NEWLINE_TOKEN Please help me . NEWLINE_TOKEN Sorry , Ma'am . NEWLINE_TOKEN You should have
--------------------------------------------------
Reference:
 been on time . NEWLINE_TOKEN I know , I am so sorry ! NEWLINE_TOKEN I got stuck in traffic . NEWLINE_TOKEN You know how it is . NEWLINE_TOKEN Just , please ! Can you do this ? NEWLINE_TOKEN Okay , hold on . NEWLINE_TOKEN Thank you . NEWLINE_TOKEN Window seat , please . NEWLINE_TOKEN Excuse me . NEWLINE_TOKEN `` Oh you ! '' NEWLINE_TOKEN `` Oh you ! '' NEWLINE_TOKEN `` Entry to Delhi . '' NEWLINE_TOKEN `` Oh you ! Entry to Delhi . '' NEWLINE_TOKEN `` Oh you ! Entry to Delhi . Oh welcome . '' NEWLINE_TOKEN `` Delhi is f
--------------------------------------------------
Generated:
any 

## RNN (LSTM + Word2vec) with tensorflow

https://www.tensorflow.org/text/tutorials/text_generation

### Model

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense
from tensorflow.keras.layers import LSTM

model = Sequential([
    Embedding(
        input_dim=len(word_to_idx),
        output_dim=100,
        weights=[embedding_matrix],
        trainable=True
    ),
    LSTM(128),
    Dense(
        units=len(word_to_idx),
        activation='softmax'
    )
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


2025-05-07 12:54:34.744369: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746615274.882922    6517 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746615274.922953    6517 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1746615275.262486    6517 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746615275.262513    6517 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1746615275.262515    6517 computation_placer.cc:177] computation placer alr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     5,867,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,867,900 (22.38 MB)

 Trainable params: 5,867,900 (22.38 MB)

 Non-trainable params: 0 (0.00 B)

### Train

In [10]:
model.fit(X_train, y_train, batch_size=64, epochs=5)

model.save("lstm_movie_scripts.h5")

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 138s 174ms/step - accuracy: 0.1490 - loss: 6.3452
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 123s 157ms/step - accuracy: 0.2669 - loss: 4.4034
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 120s 154ms/step - accuracy: 0.2920 - loss: 3.9569
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 115s 147ms/step - accuracy: 0.3080 - loss: 3.7329
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 122s 155ms/step - accuracy: 0.3261 - loss: 3.5145


### Text generation

In [ ]:
import tensorflow as tf
import numpy as np

def generate_text_lstm(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0):
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()

    for _ in range(max_len):
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices

        # Tensor sur GPU
        input_tensor = tf.convert_to_tensor([input_indices], dtype=tf.int32)  # shape: (1, seq_len)

        # Prédiction
        preds = model(input_tensor, training=False)[0].numpy()  # logits shape: (vocab_size,)
        preds = np.log(preds + 1e-8) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)

        next_idx = np.random.choice(len(preds), p=preds)
        next_word = idx_to_word.get(next_idx, "<UNK>")

        if next_word == "<PAD>":
            break

        tokens.append(next_word)
        generated.append(next_word)

    return " ".join(generated)


### Scoring

In [24]:
from bert_score import BERTScorer

device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

bleu_scores, rouge_scores, bert_scores = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 20
GENERATE_LIMIT_SIZE = 50

scorer = BERTScorer(lang="en", rescale_with_baseline=True, model_type="bert-base-uncased")

print("Début de l’évaluation...")

total = len(df_test)
i = 1

for script in df_test["Script"]:
    print(f"Script {i}/{total}")
    # eg. script = "This is a test script. It contains multiple lines.\nThis is the second line."
    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)
    # seed_text = "This is a test script. It contains multiple lines."

    generated_text = generate_text_lstm(model, seed_text, word_to_idx, idx_to_word,
                                   max_len=GENERATE_LIMIT_SIZE,
                                   seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
                                   temperature=0.8)
    # generated_text = "This is a test script. It contains multiple lines.\n This is generated text."

    reference = " ".join(custom_tokenize(script))
    generated = " ".join(custom_tokenize(generated_text))

    reference = reference[len(seed_text):len(seed_text) + len(generated)]
    generated = generated[len(seed_text):len(seed_text) + len(generated)]
    
    # print(f"Seed:\n{seed_text}")
    # print("-" * 50)
    # print(f"Reference:\n{reference}")
    # print("-" * 50)
    # print(f"Generated:\n{generated}")
    # print("-" * 50)
    # reference = "It contains multiple lines.\n This is the second line."

    bleu, rouge = calculate_scores(reference, generated)
    bleu_scores.append(bleu)
    rouge_scores.append(rouge)

    P, R, F1 = scorer.score([generated], [reference])
    bert_score = F1.mean().item()
    bert_scores.append(bert_score)
    i += 1

print("=" * 50)
print("Moyenne des scores BLEU :", sum(bleu_scores) / len(bleu_scores))
print("Moyenne des scores ROUGE :", sum(rouge_scores) / len(rouge_scores))
print("Moyenne des scores BERT :", sum(bert_scores) / len(bert_scores))


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Début de l’évaluation...
Script 1/200
Script 2/200
Script 3/200
Script 4/200
Script 5/200
Script 6/200
Script 7/200
Script 8/200
Script 9/200
Script 10/200
Script 11/200
Script 12/200
Script 13/200
Script 14/200
Script 15/200
Script 16/200
Script 17/200
Script 18/200
Script 19/200
Script 20/200
Script 21/200
Script 22/200
Script 23/200
Script 24/200
Script 25/200
Script 26/200
Script 27/200
Script 28/200
Script 29/200
Script 30/200
Script 31/200
Script 32/200
Script 33/200
Script 34/200
Script 35/200
Script 36/200
Script 37/200
Script 38/200
Script 39/200
Script 40/200
Script 41/200
Script 42/200
Script 43/200
Script 44/200
Script 45/200
Script 46/200
Script 47/200
Script 48/200
Script 49/200
Script 50/200
Script 51/200
Script 52/200
Script 53/200
Script 54/200
Script 55/200
Script 56/200
Script 57/200
Script 58/200
Script 59/200
Script 60/200
Script 61/200
Script 62/200
Script 63/200
Script 64/200
Script 65/200
Script 66/200
Script 67/200
Script 68/200
Script 69/200
Script 70/200
Scri

## RNN (GRU Bidirectional + Word2vec)
### Model

In [11]:
import torch.nn as nn

class ScriptRNNBI(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        self.rnn = nn.GRU(
            input_size=embedding_matrix.size(1),
            hidden_size=hidden_size,
            batch_first=True,
            num_layers=2,
            dropout=0.2,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size * 2, embedding_matrix.size(0))

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


### Train

In [28]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi = ScriptRNNBI(embedding_matrix, hidden_size=64).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word_to_idx['<PAD>'])
optimizer = torch.optim.Adam(model_bi.parameters(), lr=0.001)

from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_model.pth"):
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("New best model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

train_model(model_bi, train_loader, criterion, optimizer,
            num_epochs=10, patience=3, save_path="best_script_rnn_bi.pth")

Epoch 1/10 | Loss: 4.9141
New best model saved.
Epoch 2/10 | Loss: 3.9175
New best model saved.
Epoch 3/10 | Loss: 3.6080
New best model saved.
Epoch 4/10 | Loss: 3.3687
New best model saved.
Epoch 5/10 | Loss: 3.1675
New best model saved.
Epoch 6/10 | Loss: 2.9924
New best model saved.
Epoch 7/10 | Loss: 2.8342
New best model saved.
Epoch 8/10 | Loss: 2.6948
New best model saved.
Epoch 9/10 | Loss: 2.5696
New best model saved.
Epoch 10/10 | Loss: 2.4552
New best model saved.


### Load model

In [16]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi = ScriptRNNBI(embedding_matrix, hidden_size=64).to(device)


model_bi.load_state_dict(torch.load("best_script_rnn_bi.pth"))
model_bi.eval()

import torch.nn.functional as F

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0, device="cpu"):
    model.eval()
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()
    for _ in range(max_len):
        # Context = last seq_len tokens of the generated text
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
            
        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        next_word = idx_to_word.get(next_idx, "<UNK>")

        if next_word == "<PAD>": break
        tokens.append(next_word)
        generated.append(next_word)
    return " ".join(generated)

generated_text = generate_text(
    model=model_bi,
    seed_text="What's going on?",
    word_to_idx=word_to_idx,
    idx_to_word=idx_to_word,
    max_len=50,
    seq_len=20,
    device=device,
    temperature=0.8
)

print("Texte généré :\n", generated_text)


Texte généré :
 What 's going on ? NEWLINE_TOKEN Look at them . NEWLINE_TOKEN - Ok , it 's good . NEWLINE_TOKEN - Okay , guys . No , no , no . I 'm too , okay ? NEWLINE_TOKEN - I 'm gon na be covered . NEWLINE_TOKEN Why do n't be silly , be real .


### Evaluation

In [17]:
from bert_score import BERTScorer

bleu_scores, rouge_scores, bert_scores = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_SIZE = 100

scorer = BERTScorer(lang="en", rescale_with_baseline=True, model_type="bert-base-uncased")

total = len(df_test)
i = 1

print("Evaluation of ScriptRNNBI (GRU bidirectionnel)...")
for script in df_test["Script"]:
    print(f"Script {i}/{total}")
    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)

    generated_text = generate_text(
        model=model_bi,
        seed_text=seed_text,
        word_to_idx=word_to_idx,
        idx_to_word=idx_to_word,
        max_len=GENERATE_LIMIT_SIZE,
        seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
        device=device,
        temperature=0.8
    )

    # Séparer partie générée
    generated_tokens = custom_tokenize(generated_text)
    new_tokens = generated_tokens[len(seed_tokens):]
    generated = " ".join(new_tokens)

    reference_tokens = custom_tokenize(script)
    reference = " ".join(reference_tokens)[len(seed_text):len(seed_text) + len(generated)]

    # Scores
    bleu, rouge = calculate_scores(reference, generated)
    P, R, F1 = scorer.score([generated], [reference])
    bert = F1.mean().item()

    bleu_scores.append(bleu)
    rouge_scores.append(rouge)
    bert_scores.append(bert)

    i += 1

# Moyennes
print("=" * 50)
print(f"Average BLEU Score : {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE Score : {sum(rouge_scores)/len(rouge_scores):.4f}")
print(f"Average BERT Score : {sum(bert_scores)/len(bert_scores):.4f}")


Evaluation of ScriptRNNBI (GRU bidirectionnel)...
Script 1/200
Script 2/200
Script 3/200
Script 4/200
Script 5/200
Script 6/200
Script 7/200
Script 8/200
Script 9/200
Script 10/200
Script 11/200
Script 12/200
Script 13/200
Script 14/200
Script 15/200
Script 16/200
Script 17/200
Script 18/200
Script 19/200
Script 20/200
Script 21/200
Script 22/200
Script 23/200
Script 24/200
Script 25/200
Script 26/200
Script 27/200
Script 28/200
Script 29/200
Script 30/200
Script 31/200
Script 32/200
Script 33/200
Script 34/200
Script 35/200
Script 36/200
Script 37/200
Script 38/200
Script 39/200
Script 40/200
Script 41/200
Script 42/200
Script 43/200
Script 44/200
Script 45/200
Script 46/200
Script 47/200
Script 48/200
Script 49/200
Script 50/200
Script 51/200
Script 52/200
Script 53/200
Script 54/200
Script 55/200
Script 56/200
Script 57/200
Script 58/200
Script 59/200
Script 60/200
Script 61/200
Script 62/200
Script 63/200
Script 64/200
Script 65/200
Script 66/200
Script 67/200
Script 68/200
Script 

## RNN (LSTM Bidirectional + Word2vec)
### Model

In [18]:
import torch.nn as nn

class ScriptRNNBI_LSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        self.rnn = nn.LSTM(
            input_size=embedding_matrix.size(1),
            hidden_size=hidden_size,
            batch_first=True,
            num_layers=2,
            dropout=0.2,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size * 2, embedding_matrix.size(0))

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


### Train

In [20]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi = ScriptRNNBI_LSTM(embedding_matrix, hidden_size=64).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word_to_idx['<PAD>'])
optimizer = torch.optim.Adam(model_bi.parameters(), lr=0.001)

from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

def train_model(model, train_loader, criterion, optimizer, num_epochs=10, patience=3, save_path="best_model.pth"):
    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print("New best model saved.")
        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

train_model(model_bi, train_loader, criterion, optimizer,
            num_epochs=10, patience=3, save_path="best_script_rnn_bi_lstm.pth")

Epoch 1/10 | Loss: 5.0474
New best model saved.
Epoch 2/10 | Loss: 4.0786
New best model saved.
Epoch 3/10 | Loss: 3.7885
New best model saved.
Epoch 4/10 | Loss: 3.5826
New best model saved.
Epoch 5/10 | Loss: 3.4098
New best model saved.
Epoch 6/10 | Loss: 3.2581
New best model saved.
Epoch 7/10 | Loss: 3.1210
New best model saved.
Epoch 8/10 | Loss: 2.9896
New best model saved.
Epoch 9/10 | Loss: 2.8739
New best model saved.
Epoch 10/10 | Loss: 2.7690
New best model saved.


### Load model

In [21]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")

model_bi_lstm = ScriptRNNBI_LSTM(embedding_matrix, hidden_size=64).to(device)


model_bi_lstm.load_state_dict(torch.load("best_script_rnn_bi_lstm.pth"))
model_bi_lstm.eval()

import torch.nn.functional as F

def generate_text(model, seed_text, word_to_idx, idx_to_word, max_len=50, seq_len=20, temperature=1.0, device="cpu"):
    model.eval()
    tokens = custom_tokenize(seed_text)
    generated = tokens.copy()
    for _ in range(max_len):
        # Context = last seq_len tokens of the generated text
        context = tokens[-seq_len:]
        input_indices = [word_to_idx.get(w, word_to_idx["<UNK>"]) for w in context]

        if len(input_indices) < seq_len:
            input_indices = [word_to_idx["<PAD>"]] * (seq_len - len(input_indices)) + input_indices
            
        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_tensor)
            probs = F.softmax(logits[0] / temperature, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()

        next_word = idx_to_word.get(next_idx, "<UNK>")

        if next_word == "<PAD>": break
        tokens.append(next_word)
        generated.append(next_word)
    return " ".join(generated)

generated_text = generate_text(
    model=model_bi_lstm,
    seed_text="What's going on?",
    word_to_idx=word_to_idx,
    idx_to_word=idx_to_word,
    max_len=50,
    seq_len=20,
    device=device,
    temperature=0.8
)

print("Texte généré :\n", generated_text)


Texte généré :
 What 's going on ? NEWLINE_TOKEN No , but I could n't NEWLINE_TOKEN Wah it was 16 with your charms NEWLINE_TOKEN That fucker was more than . NEWLINE_TOKEN His dad would go undercover NEWLINE_TOKEN just had to push . NEWLINE_TOKEN The car was vascular ones NEWLINE_TOKEN and is due to create fate . NEWLINE_TOKEN We


### Evaluation

In [ ]:
from bert_score import BERTScorer

bleu_scores, rouge_scores, bert_scores = [], [], []

SCRIPT_LIMIT_TOKEN_SIZE = 50
GENERATE_LIMIT_SIZE = 100

scorer = BERTScorer(lang="en", rescale_with_baseline=True, model_type="bert-base-uncased")

total = len(df_test)
i = 1

print("Evaluation of ScriptRNNBI (GRU bidirectionnel)...")
for script in df_test["Script"]:
    print(f"Script {i}/{total}")
    seed_tokens = custom_tokenize(script)[:SCRIPT_LIMIT_TOKEN_SIZE]
    seed_text = " ".join(seed_tokens)

    generated_text = generate_text(
        model=model_bi_lstm,
        seed_text=seed_text,
        word_to_idx=word_to_idx,
        idx_to_word=idx_to_word,
        max_len=GENERATE_LIMIT_SIZE,
        seq_len=SCRIPT_LIMIT_TOKEN_SIZE,
        device=device,
        temperature=0.8
    )

    # Séparer partie générée
    generated_tokens = custom_tokenize(generated_text)
    new_tokens = generated_tokens[len(seed_tokens):]
    generated = " ".join(new_tokens)

    reference_tokens = custom_tokenize(script)
    reference = " ".join(reference_tokens)[len(seed_text):len(seed_text) + len(generated)]

    # Scores
    bleu, rouge = calculate_scores(reference, generated)
    P, R, F1 = scorer.score([generated], [reference])
    bert = F1.mean().item()

    bleu_scores.append(bleu)
    rouge_scores.append(rouge)
    bert_scores.append(bert)

    i += 1

# Moyennes
print("=" * 50)
print(f"Average BLEU Score : {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE Score : {sum(rouge_scores)/len(rouge_scores):.4f}")
print(f"Average BERT Score : {sum(bert_scores)/len(bert_scores):.4f}")


/home/rickgao/miniconda3/envs/tf-wsl/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Evaluation of ScriptRNNBI (GRU bidirectionnel)...
Script 1/200
Script 2/200
Script 3/200
Script 4/200
Script 5/200
Script 6/200
Script 7/200
Script 8/200
Script 9/200
Script 10/200
Script 11/200
Script 12/200
Script 13/200
Script 14/200
Script 15/200
Script 16/200
Script 17/200
Script 18/200
Script 19/200
Script 20/200
Script 21/200
Script 22/200
Script 23/200
Script 24/200
Script 25/200
Script 26/200
Script 27/200
Script 28/200
Script 29/200
Script 30/200
Script 31/200
Script 32/200
Script 33/200
Script 34/200
Script 35/200
Script 36/200
Script 37/200
Script 38/200
Script 39/200
Script 40/200
Script 41/200
Script 42/200
Script 43/200
Script 44/200
Script 45/200
Script 46/200
Script 47/200
Script 48/200
Script 49/200
Script 50/200
Script 51/200
Script 52/200
Script 53/200
Script 54/200
Script 55/200
Script 56/200
Script 57/200
Script 58/200
Script 59/200
Script 60/200
Script 61/200
Script 62/200
Script 63/200
Script 64/200
Script 65/200
Script 66/200
Script 67/200
Script 68/200
Script 

: 